# Exercise: Performance Optimization

Optimize the following function.

In [1]:
function work!(A, B, v)
    @assert size(A) == size(B)
    val = zero(eltype(v))
    for i in 1:N
        val = mod(v[i],256)
        A[i,1:N] = B[i,1:N] * (sin(val) * sin(val) - cos(val) * cos(val))
    end
    return A
end

work! (generic function with 1 method)

The following data is **fixed** and **not supposed to be modified**!

In [39]:
# do not modify this cell!

using Random
Random.seed!(42)

N = 8000
B = rand(N,N)
v = rand(Int, N);

const result = work!(zeros(N,N), B, v);

# do not modify this cell!

You can compare against `A_result` to test your implementation(s):

In [4]:
using Test

@test work!(zeros(N,N), B, v) ≈ result

Test Passed

You can benchmark as follows:

In [5]:
using BenchmarkTools

@btime work!(A, $B, $v) setup=(A=zeros(N,N)); # or use @benchmark for more information


  4.972 ms (17668 allocations: 10.17 MiB)


## Your Optimizations

Your optimized variants go here!

**Hints** (hopefully):
* What is suboptimal about the code? What is it that you'd want to change (but can't directly)?
* Sometimes writing the code in a different way doesn't give direct speedups but enables further optimization.
* A ~30x speedup should be possible on most systems 😉

# Your variants go here...

Problems that I see:
- uses slices that allocate
- uses row operations, `B[i,1:N]`
- you can probably do something with broadcasting
- we are evaluating sin(val) and cos(val) multiple times

In [6]:
# adding views to reduce allocations of the slice
function work_views!(A, B, v)
    @assert size(A) == size(B)
    val = zero(eltype(v))
    for i in 1:N
        val = mod(v[i],256)
        @views A[i,1:N] = B[i,1:N] * (sin(val) * sin(val) - cos(val) * cos(val))
    end
    return A
end
@test work_views!(zeros(N,N), B, v) ≈ result

# NOTES: There are two for loops here, the hidden one is the 1:N

Test Passed

In [7]:
@btime work_views!(A, $B, $v) setup=(A=zeros(N,N));

  4.271 ms (16068 allocations: 5.28 MiB)


In [23]:
# fixing the calculation of val every time
function work_views_2times!(A, B, v)
    @assert size(A) == size(B)
    val = zero(eltype(v))
    for i in 1:N
        val = mod(v[i],256)
        @views A[i,1:N] = B[i,1:N] * (sin(val)^2 - cos(val)^2)
    end
    return A
end
@test work_views_2times!(zeros(N,N), B, v) ≈ result

Test Passed

In [24]:
@btime work_views_2times!(A, $B, $v) setup=(A=zeros(N,N));

  4.192 ms (14468 allocations: 5.26 MiB)


In [37]:
# can we vectorize the v so that we can go column by column?
function work_views_2times_vec!(A, B, v)
    @assert size(A) == size(B)
    val_vec = mod.(v,256)
    @views A .= B .* (sin.(val_vec).^2 - cos.(val_vec).^2)
    return A
end
@test work_views_2times_vec!(zeros(N,N), B, v) ≈ result


Test Passed

In [38]:
@btime work_views_2times_vec!(A, $B, $v) setup=(A=zeros(N,N));

  505.562 μs (12 allocations: 25.28 KiB)


In [40]:
# Solutions
@btime work!(A, $B, $v) setup=(A=zeros(N,N)); # or use @benchmark for more information
@btime work_views!(A, $B, $v) setup=(A=zeros(N,N));
@btime work_views_2times!(A, $B, $v) setup=(A=zeros(N,N));
@btime work_views_2times_vec!(A, $B, $v) setup=(A=zeros(N,N));


  1.032 s (190468 allocations: 980.81 MiB)
  965.192 ms (174468 allocations: 492.47 MiB)
  968.736 ms (158468 allocations: 492.23 MiB)
  67.083 ms (12 allocations: 250.28 KiB)


In [54]:
# trying two loops
function work_views_2times_2loops!(A, B, v)
    @assert size(A) == size(B)
    val = zero(eltype(v))
    comp_val = similar(v);
    for i in 1:N
        val = mod(v[i],256)   
        comp_val[i] = (sin(val)^2 - cos(val)^2)
    end
    
    for j in 1:N
        for i in 1:N                 
            @views A[i,j] .= B[i,j] .* comp_val[i] 
        end
    end
    return A
end

# work_views_2times_2loops!(zeros(N,N), B, v)
# @test work_views_2times_2loops!(zeros(N,N), B, v) ≈ result

Int64

LoadError: InexactError: Int64(-0.13673721820783358)

## Bonus Question: Performance limit?

Look at your final optimized version of `work!`.

* In the limit of larger `A` and `B`, what is conceptually limiting the performance, the compute capability or memory transfer (i.e. reading and writing `A` and `B`)?

Let's try to quickly estimate the maximal memory bandwidth that a single-CPU core can achieve on the given computer:

In [ ]:
using STREAMBenchmark
membw = memory_bandwidth(; nthreads=1).median / 1000 # memory bandwidth in GB /s

For references, a single CPU-core in [Noctua 2](https://pc2.uni-paderborn.de/systems-and-services/noctua-2) can achieve a **maximal memory bandwidth of ~45 GB/s**.

* Given the maximal memory bandwidth, can you give a performance bound estimate, i.e. the minimal runtime that we could possibly hope to achieve?
  * Hint: how many flops are performed per iteration and how many bytes are transferred?
* How far off is your implementation from achieving the limit (in percent)?

In [ ]:
# Your computation goes here...